# COMP64702 RAG Coursework
## Notebook 2: RAG Pipeline

This notebook implements the Retrieval-Augmented Generation (RAG) pipeline,
including hybrid retrieval and answer generation.


| Component | Original | Improved |
|-----------|----------|----------|
| **Chunking** | Section-as-chunk (inconsistent sizes) | Sentence-level with 100-char overlap |
| **Embedding model** | `all-MiniLM-L6-v2` (generic) | `BAAI/bge-small-en-v1.5` (retrieval-optimised) |
| **Query encoding** | No prefix | BGE query prefix for asymmetric retrieval |
| **Chunk text format** | `[section] text` | `[title | section] text` |
| **Retrieval** | RRF hybrid (dense + BM25) | RRF hybrid + cross-encoder reranking |
| **Generation** | 80 max tokens, basic prompt | 150 max tokens, structured prompt with recency ordering |
| **Inference loop** | Silent | Per-query progress logging |
| **Corpus path** | Hard-coded original | Falls back gracefully to original if expanded not found |

In [1]:
# Install dependencies
# CHANGE: combined into one pip call with -q for cleaner output
!pip install sentence-transformers transformers rank-bm25 accelerate -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
# Imports
# CHANGE: added `re` (sentence splitting) and `CrossEncoder` (reranking);
#         removed unused `math` and `List` imports
import json
import re
from pathlib import Path
from dataclasses import dataclass
from typing import List
import numpy as np
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
from rank_bm25 import BM25Okapi
import torch

print("Imports OK")

Imports OK


In [3]:
# Paths and data loading
# CHANGE: added fallback to background_corpus_expanded.json if it exists,
#         otherwise uses the original corpus (no breaking change).

DATA_DIR = Path(".")

expanded_corpus_path = DATA_DIR / "background_corpus_expanded.json"
original_corpus_path = DATA_DIR / "background_corpus.json"

corpus_path = expanded_corpus_path if expanded_corpus_path.exists() else original_corpus_path
query_path  = DATA_DIR / "benchmark_input_only.json"
output_path = DATA_DIR / "test_outputs.json"

with open(corpus_path, "r", encoding="utf-8") as f:
    corpus = json.load(f)

with open(query_path, "r", encoding="utf-8") as f:
    query_payload = json.load(f)

queries = query_payload["queries"]

print(f"Using corpus: {corpus_path.name}")
print(f"Loaded {len(corpus)} documents and {len(queries)} queries")

Using corpus: background_corpus_expanded.json
Loaded 23 documents and 13 queries


In [4]:
# Dataclasses (unchanged)

@dataclass
class Chunk:
    doc_id: str
    title: str
    section: str
    text: str

@dataclass
class RetrievalResult:
    chunk: Chunk
    score: float

In [5]:
# IMPROVED CHUNKING
#
# Original problem: entire sections became single chunks, leading to very
# inconsistent chunk sizes (some 3000+ chars, some 50 chars). Large chunks
# dominate embeddings; tiny chunks lose context.
#
# Fix: sentence-level splitting with 100-char overlap so that context from
# the previous chunk carries over, and every chunk stays within a manageable
# size for the embedding model.

def chunk_documents(corpus_docs, max_chars=500, overlap_chars=100):
    """
    Split documents into sentence-level chunks with overlap.

    Args:
        max_chars:     Target maximum characters per chunk.
        overlap_chars: Characters carried over from the previous chunk
                       to preserve cross-boundary context.

    Original behaviour (section-as-chunk, no overlap) is equivalent to
    calling this with max_chars=infinity — kept here for reference only.
    """
    chunks = []

    for doc in corpus_docs:
        doc_id   = doc.get("doc_id", "")
        title    = doc.get("title", "")
        sections = doc.get("sections", [])

        if sections:
            for sec in sections:
                heading      = sec.get("heading", "Section")
                content_list = sec.get("content", [])
                full_section = " ".join(content_list).strip()
                if not full_section:
                    continue

                # Split on sentence boundaries
                sentences = re.split(r'(?<=[.!?])\s+', full_section)

                current_chunk = ""
                for sent in sentences:
                    sent = sent.strip()
                    if not sent:
                        continue

                    if len(current_chunk) + len(sent) + 1 <= max_chars:
                        current_chunk = (current_chunk + " " + sent).strip()
                    else:
                        if current_chunk:
                            chunks.append(Chunk(
                                doc_id=doc_id, title=title,
                                section=heading, text=current_chunk
                            ))
                        # Carry overlap from end of previous chunk
                        overlap_text = (
                            current_chunk[-overlap_chars:]
                            if len(current_chunk) > overlap_chars
                            else current_chunk
                        )
                        current_chunk = (overlap_text + " " + sent).strip()

                if current_chunk:
                    chunks.append(Chunk(
                        doc_id=doc_id, title=title,
                        section=heading, text=current_chunk
                    ))
        else:
            # Fallback for docs without sections
            full_text = doc.get("full_text", "").strip()
            if full_text:
                sentences = re.split(r'(?<=[.!?])\s+', full_text)
                current_chunk = ""
                for sent in sentences:
                    sent = sent.strip()
                    if not sent:
                        continue
                    if len(current_chunk) + len(sent) + 1 <= max_chars:
                        current_chunk = (current_chunk + " " + sent).strip()
                    else:
                        if current_chunk:
                            chunks.append(Chunk(
                                doc_id=doc_id, title=title,
                                section="Full text", text=current_chunk
                            ))
                        overlap_text = (
                            current_chunk[-overlap_chars:]
                            if len(current_chunk) > overlap_chars
                            else current_chunk
                        )
                        current_chunk = (overlap_text + " " + sent).strip()
                if current_chunk:
                    chunks.append(Chunk(
                        doc_id=doc_id, title=title,
                        section="Full text", text=current_chunk
                    ))

    return chunks


chunks = chunk_documents(corpus)
print(f"Produced {len(chunks)} chunks")

# Show chunk size distribution
lengths = [len(c.text) for c in chunks]
print(f"Chunk size — min: {min(lengths)}  max: {max(lengths)}  mean: {int(np.mean(lengths))}")

Produced 486 chunks
Chunk size — min: 34  max: 967  mean: 372


In [6]:
# IMPROVED EMBEDDING MODEL
#
# Original: all-MiniLM-L6-v2  (generic, trained on NLI/STS tasks)
# Improved: BAAI/bge-small-en-v1.5
#   - Same 384 dimensions → identical memory and speed profile
#   - Trained specifically for retrieval (top MTEB performer at this size)
#   - Requires a query prefix for asymmetric retrieval (corpus side uses no prefix)

EMBED_MODEL_NAME = "BAAI/bge-small-en-v1.5"
embedding_model  = SentenceTransformer(EMBED_MODEL_NAME)

# BGE asymmetric retrieval: queries use this prefix, corpus chunks do not
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

print(f"Embedding model loaded: {EMBED_MODEL_NAME}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded: BAAI/bge-small-en-v1.5


In [7]:
# Embed all chunks
# CHANGE: chunk text format updated from "[section] text"
#         to "[title | section] text" so title context is always present.
#         No query prefix on the corpus side (BGE asymmetric retrieval).

chunk_texts = [f"[{c.title} | {c.section}] {c.text}" for c in chunks]

chunk_embeddings = embedding_model.encode(
    chunk_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=32
)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Chunk embeddings shape: (486, 384)


In [8]:
# Build BM25 retriever (unchanged)

tokenized_chunks = [text.lower().split() for text in chunk_texts]
bm25 = BM25Okapi(tokenized_chunks)
print("BM25 retriever ready")

BM25 retriever ready


In [9]:
# Dense retrieval function
# CHANGE: queries are now encoded with the BGE query prefix for correct
#         asymmetric retrieval (corpus chunks are encoded without it).

def dense_retrieve(query, top_k=10):
    prefixed_query = QUERY_PREFIX + query
    query_emb = embedding_model.encode([prefixed_query], convert_to_numpy=True)[0]

    chunk_norms = np.linalg.norm(chunk_embeddings, axis=1)
    query_norm  = np.linalg.norm(query_emb)
    sims = (chunk_embeddings @ query_emb) / (chunk_norms * query_norm + 1e-12)

    top_indices = np.argsort(sims)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(sims[i])) for i in top_indices]

In [10]:
# BM25 retrieval function (unchanged logic)

def bm25_retrieve(query, top_k=10):
    tokenized_query = query.lower().split()
    scores = bm25.get_scores(tokenized_query)
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [RetrievalResult(chunk=chunks[i], score=float(scores[i])) for i in top_indices]

In [11]:
# Hybrid retrieval function (RRF fusion)
# CHANGE: deduplication key now uses text[:50] instead of the full text
#         to avoid hash misses on chunks with trailing whitespace differences.

def hybrid_retrieve(query, top_k=10, rrf_k=60):
    dense_results = dense_retrieve(query, top_k=top_k * 2)
    bm25_results  = bm25_retrieve(query,  top_k=top_k * 2)

    score_map = {}

    for rank, res in enumerate(dense_results, start=1):
        key = (res.chunk.doc_id, res.chunk.section, res.chunk.text[:50])
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (rrf_k + rank)

    for rank, res in enumerate(bm25_results, start=1):
        key = (res.chunk.doc_id, res.chunk.section, res.chunk.text[:50])
        score_map.setdefault(key, {"chunk": res.chunk, "score": 0.0})
        score_map[key]["score"] += 1.0 / (rrf_k + rank)

    merged = [
        RetrievalResult(chunk=v["chunk"], score=v["score"])
        for v in score_map.values()
    ]
    merged.sort(key=lambda x: x.score, reverse=True)
    return merged[:top_k]

In [12]:
# NEW: Cross-encoder reranker
#
# Original pipeline relied purely on RRF hybrid scores. A cross-encoder
# reads the (query, chunk) pair jointly, capturing relevance signals that
# the bi-encoder misses (e.g. exact phrase overlap, negation, specificity).
#
# Strategy:
#   1. hybrid_retrieve() fetches top-15 candidates (broad recall)
#   2. Cross-encoder rescores all 15
#   3. Return top-5 by reranker score

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
reranker = CrossEncoder(RERANKER_MODEL)
print(f"Reranker loaded: {RERANKER_MODEL}")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Reranker loaded: cross-encoder/ms-marco-MiniLM-L-6-v2


In [13]:
# NEW: retrieve_and_rerank — full retrieval pipeline
#
# Replaces direct calls to hybrid_retrieve() in the inference loop.
# The original hybrid_retrieve() is still available for ablation or comparison.

def retrieve_and_rerank(query, top_k=5, candidate_k=15):
    """
    Full retrieval pipeline:
      hybrid retrieval (candidate_k) -> cross-encoder reranking -> top_k
    """
    candidates = hybrid_retrieve(query, top_k=candidate_k)
    if not candidates:
        return []

    pairs  = [(query, r.chunk.text) for r in candidates]
    scores = reranker.predict(pairs)

    reranked = sorted(
        zip(scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )
    return [res for _, res in reranked[:top_k]]


# Quick sanity check
test_results = retrieve_and_rerank("What is biryani?", top_k=3)
for r in test_results:
    print(f"  [{r.chunk.doc_id} | {r.chunk.section}] {r.chunk.text[:80]}...")

  [wiki_biryani | Introduction] Biryani is a mixed rice dish originating in South Asia, traditionally made with ...
  [wiki_biryani | In the Indian subcontinent] There are many types of biryani in the Indian subcontinent. Biryani is the singl...
  [wiki_biryani | Introduction] countries, often with local variations, and often brought there by South Asian d...


In [14]:
# Load Qwen model (unchanged)

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    torch_dtype="auto",
    device_map="auto"
)
print("Qwen loaded")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Qwen loaded


In [15]:
# IMPROVED PROMPT BUILDER
#
# Changes vs original:
#   1. Context chunks ordered with MOST relevant LAST — transformers have a
#      recency bias, so the final chunk has stronger influence on generation.
#      Reranked results arrive best-first, so we reverse before building context.
#   2. Each chunk is labelled with its source title (not just section).
#   3. System prompt is tighter and more direct.
#   4. Removed "Do not mention context/documents" — this instruction confused
#      the model and caused it to omit useful attribution phrases.

def build_messages(query, retrieved_results):
    # Reverse so MOST relevant chunk appears last (recency bias optimisation)
    ordered = list(reversed(retrieved_results))

    context_parts = [
        f"[{r.chunk.title}]\n{r.chunk.text}"
        for r in ordered
    ]
    context = "\n\n".join(context_parts)

    return [
        {
            "role": "system",
            "content": (
                "You are a South Asian cuisine expert. "
                "Answer the question using ONLY the provided context. "
                "Be specific and factual. "
                "If the context does not contain enough information, say so briefly."
            )
        },
        {
            "role": "user",
            "content": (
                f"Context:\n{context}\n\n"
                f"Question: {query}\n\n"
                "Answer in 2 to 3 sentences based on the context above."
            )
        }
    ]

In [16]:
# IMPROVED GENERATION FUNCTION
#
# CHANGE: max_new_tokens increased from 80 to 150.
# The original value cut off many answers mid-sentence (e.g. sa_005 ended
# with "quickly lose" — visibly truncated). 150 gives room for comparative
# or multi-part answers without padding short, easy ones.

def generate_answer(messages, max_new_tokens=150):
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    response  = tokenizer.decode(generated, skip_special_tokens=True)
    return response.strip()

In [17]:
# Test on the two queries that previously failed
# CHANGE: expanded from a single sample test to three representative queries,
#         including the ones that were truncated or misanswered in the original.

test_queries = [
    ("sa_000", "What is biryani and which region is it originally associated with?"),
    ("sa_001", "What distinguishes Hyderabadi biryani from Lucknowi (Awadhi) biryani?"),
    ("sa_009", "What is butter chicken and where did it originate?"),
]

for qid, q in test_queries:
    retrieved = retrieve_and_rerank(q, top_k=5)
    messages  = build_messages(q, retrieved)
    answer    = generate_answer(messages)
    print(f"\n{'─'*60}")
    print(f"[{qid}] {q}")
    print(f"Top doc: {retrieved[0].chunk.doc_id if retrieved else 'NONE'}")
    print(f"ANSWER: {answer}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



────────────────────────────────────────────────────────────
[sa_000] What is biryani and which region is it originally associated with?
Top doc: wiki_biryani
ANSWER: Biryani is a mixed rice dish originating in South Asia, traditionally made with rice, meat (chicken, goat, beef) or seafood (prawns or fish), vegetables, and spices. It was once associated with the region's Muslim population but is now a mainstream culinary staple embraced by every demographic.

────────────────────────────────────────────────────────────
[sa_001] What distinguishes Hyderabadi biryani from Lucknowi (Awadhi) biryani?
Top doc: wiki_hyderabadi_biryani
ANSWER: Hyderabadi biryani is distinguished from Lucknowi (Awadhi) biryani by its incorporation of Deccani or Telangana flavors into the dish.

────────────────────────────────────────────────────────────
[sa_009] What is butter chicken and where did it originate?
Top doc: wiki_butter_chicken
ANSWER: Butter chicken is a curry made from chicken cooked in a spic

In [18]:
# Full inference loop
# CHANGE: uses retrieve_and_rerank() instead of hybrid_retrieve();
#         added per-query progress logging.

results = []

for i, item in enumerate(queries):
    query_id = item["query_id"]
    query    = item["query"]
    print(f"[{i+1}/{len(queries)}] {query_id}: {query[:60]}...")

    retrieved = retrieve_and_rerank(query, top_k=5)
    messages  = build_messages(query, retrieved)
    response  = generate_answer(messages)

    results.append({
        "query_id": query_id,
        "query":    query,
        "response": response,
        "retrieved_context": [
            {
                "doc_id": r.chunk.doc_id,
                "text":   f"[{r.chunk.section}] {r.chunk.text[:500]}"
            }
            for r in retrieved
        ]
    })

print(f"\nGenerated answers for {len(results)} queries")

[1/13] sa_000: What is biryani and which region is it originally associated...
[2/13] sa_001: What distinguishes Hyderabadi biryani from Lucknowi (Awadhi)...
[3/13] sa_002: What is the difference between naan and roti?...
[4/13] sa_003: How is dosa batter traditionally prepared?...
[5/13] sa_004: What ingredients are used in a classic samosa and how is it ...
[6/13] sa_005: What is cardamom and what are the differences between green ...
[7/13] sa_006: How does Sri Lankan cuisine differ from South Indian cuisine...
[8/13] sa_007: What is the difference between a curry and a masala?...
[9/13] sa_008: What are the key features of Pakistani cuisine compared to N...
[10/13] sa_009: What is butter chicken and where did it originate?...
[11/13] sa_010: How is tandoori chicken traditionally prepared?...
[12/13] sa_011: What is dal and why is it important in South Asian cuisine?...
[13/13] sa_012: What is the difference between biryani and pilaf?...

Generated answers for 13 queries


In [19]:
# Save outputs (unchanged)

with open(output_path, "w", encoding="utf-8") as f:
    json.dump({"results": results}, f, indent=2, ensure_ascii=False)

print(f"Saved outputs to {output_path}")

Saved outputs to test_outputs.json


In [20]:
# Demo: test a custom query and inspect retrieved context
# CHANGE: now uses retrieve_and_rerank() and labels retrieved chunks
#         as "after reranking" for clarity.

custom_query = input("Enter your question: ")

retrieved = retrieve_and_rerank(custom_query, top_k=5)
messages  = build_messages(custom_query, retrieved)
response  = generate_answer(messages)

print("\nANSWER:")
print(response)

print("\nRETRIEVED CONTEXT (after reranking):")
for i, r in enumerate(retrieved, 1):
    print(f"\n{i}. [{r.chunk.doc_id} | {r.chunk.section}]")
    print(r.chunk.text[:300])


ANSWER:
Chole bhature is a South Asian dish consisting of a batter filled with chole (tandoori chicken) and served with steamed rice. It's typically prepared in a tandoor oven and can be enjoyed either hot or cold.

RETRIEVED CONTEXT (after reranking):

1. [wiki_nepali_cuisine | Staple foods]
Dal bhat is one of the best-known staple meals, usually consisting of rice, lentil soup, and side dishes. Vegetables, pickles, flatbreads, and dumplings such as momo are also common.

2. [wiki_mughlai_cuisine | Mughal era cookery books]
lida (sweet dough), and several types of sweet dumplings, namely saṃbosa, pūrī, gulgula, and khajur. The Alwān-E-Niʿmat ("Colours of the Table") from the reign of Jahangir (r. 1605–1627), was dedicated entirely to sweetmeats. It describes nan ḵẖata'i (a biscuit-like bread, sometimes with almond: nan

3. [wiki_turmeric | Traditional uses]
( bilva ), pomegranate ( darimba ), Saraca indica , manaka ( Arum ), or manakochu , and rice paddy. The Haldi ceremony called ga

In [21]:
# Preview first 2 outputs
# CHANGE: replaced bare results[:2] slice with formatted print for readability.

for r in results[:2]:
    print(f"\n[{r['query_id']}] {r['query']}")
    print(f"ANSWER: {r['response']}")
    print(f"TOP DOC: {r['retrieved_context'][0]['doc_id']}")


[sa_000] What is biryani and which region is it originally associated with?
ANSWER: Biryani is a mixed rice dish originating in South Asia, traditionally made with rice, meat (chicken, goat, beef) or seafood (prawns or fish), vegetables, and spices. It was once associated with the region's Muslim population but is now a mainstream culinary staple embraced by every demographic.
TOP DOC: wiki_biryani

[sa_001] What distinguishes Hyderabadi biryani from Lucknowi (Awadhi) biryani?
ANSWER: Hyderabadi biryani is distinguished from Lucknowi (Awadhi) biryani by its incorporation of Deccani or Telangana flavors into the dish.
TOP DOC: wiki_hyderabadi_biryani
